In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 02 — graph/processing 模块测试

覆盖 `backend/graph/processing/` 的完整处理链路：
1. 文件读取 → 分块
2. 图结构构建
3. 实体关系提取 → 写入 Neo4j
4. 生成 Entity Embedding
5. 相似实体检测 (GDS)
6. 实体合并
7. 消歧 + 对齐

**前置条件**：Neo4j 已启动，环境变量已配置。

In [15]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.pipelines.document_processor import DocumentProcessor
from backend.pipelines.models import Chunk
from backend.graph.core import connection_manager, get_connection_manager
from backend.graph.structure import GraphStructureBuilder
from backend.graph.extraction import EntityRelationExtractor, GraphWriter
from backend.graph.indexing import EntityIndexManager
from backend.graph.processing import (
    SimilarEntityDetector, GDSConfig,
    EntityMerger, EntityQualityProcessor
)
from backend.models.get_models import get_llm_model
from backend.config.settings import FILES_DIR
from backend.config.prompts.graph_prompts import (
    system_template_build_graph, human_template_build_graph,
)
from backend.config.settings import entity_types, relationship_types

print("所有模块导入成功")

所有模块导入成功


---
## Step 1：文件读取 + 分块

In [16]:
processor = DocumentProcessor()
file_chunks = processor.process(file_paths=str(FILES_DIR / "test.md"))
file_name, chunks = file_chunks[0]
print(f"文件: {file_name}")
print(f"Chunk 数量: {len(chunks)}")

[ChineseTextChunker] init 0.45066 s
处理文件: test.md (类型: .md)
DocumentProcessor找到的文件数量: 1
  F:\AllProjects\Agent\equipment-fault-qa\data\test\test.md: 2 个 Chunk
分块完成，共处理 1 个文件
文件: F:\AllProjects\Agent\equipment-fault-qa\data\test\test.md
Chunk 数量: 2


---
## Step 2：图结构构建

In [17]:
builder = GraphStructureBuilder()
builder.clear_database()
builder.create_document(type="md", uri=str(FILES_DIR), file_name=file_name, domain="test")
result = builder.create_relation_between_chunks(file_name, chunks)
print(f"Chunk 写入完成: {len(result)} 个")

创建关系耗时: 0.05秒
Chunk 写入完成: 2 个


---
## Step 3：实体关系提取

In [18]:
llm = get_llm_model()
extractor = EntityRelationExtractor(
    llm=llm,
    system_template=system_template_build_graph,
    human_template=human_template_build_graph,
    entity_types=entity_types,
    relationship_types=relationship_types,
)
file_contents = extractor.process_chunks(file_chunks)
print("实体关系提取完成")

  F:\AllProjects\Agent\equipment-fault-qa\data\test\test.md: 2 个 chunk, 缓存命中率: 100.0%
所有 chunks 处理完成, 总耗时: 0.00秒, 平均每 chunk: 0.00秒
实体关系提取完成


---
## Step 4：写入 Neo4j

In [19]:
writer = GraphWriter()
writer.process_and_write_graph_documents(file_contents)
print("GraphWriter 写入完成")

开始处理 2 个 chunks 的 GraphDocument
共处理 2 个 chunks, 有效文档 1, 错误 0
开始批量写入 1 个文档，批次大小: 10, 总批次: 1
已写入批次 1/1
开始合并 1 个唯一chunk关系
合并关系批次大小: 20, 总批次: 1
已处理合并关系批次 1/1
GraphWriter 写入完成


---
## Step 5：生成 Entity Embedding

SimilarEntityDetector 依赖实体的 `embedding` 属性来创建 GDS 投影。

In [20]:
entity_indexer = EntityIndexManager()
entity_vector = entity_indexer.create_entity_index()

graph = get_connection_manager().get_connection()
cnt = graph.query("MATCH (e:__Entity__) WHERE e.embedding IS NOT NULL RETURN count(e) AS cnt")[0]["cnt"]
print(f"有 embedding 的实体: {cnt} 个")

[transformers] The following layers were not sharded: embeddings.word_embeddings.weight, encoder.layer.*.attention.self.key.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.attention.self.value.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.self.query.bias, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.output.LayerNorm.bias, embeddings.token_type_embeddings.weight, pooler.dense.weight, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.key.bias, pooler.dense.bias, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.output.dense.bias, embeddings.position_embeddings.weight, encoder.layer.*.attention.self.value.bias, embeddings.LayerNorm.weight, embeddings.LayerNorm.bias, encoder.layer.*.attention.self.query.weight, encoder.layer.*.attention.output.dense.weight
Loading weights: 100%|██████████| 199/199 [00:00

[embedding_worker] Model loaded: shibing624/text2vec-base-chinese
已删除索引 entity_embedding（如果存在）
已删除索引 vector（如果存在）
开始为 16 个实体生成embeddings
处理实体embedding: 共16项，批次大小: 20, 总批次: 1
已处理批次 1/1, 批次耗时: 0.64秒, 平均: 0.64秒/批, 预计剩余: 0.00秒
索引创建成功，总耗时: 1.33秒
其中: embedding计算: 0.46秒, 数据库操作: 0.12秒
有 embedding 的实体: 16 个


---
## Step 6：相似实体检测

使用 GDS 进行实体投影 → KNN 相似度 → WCC 社区发现 → 候选重复组。

In [21]:
try:
    detector = SimilarEntityDetector()
    duplicates, stats = detector.process_entities()
    GDS_AVAILABLE = True
    print(f"检测完成: {len(duplicates)} 组候选重复实体")
except Exception as e:
    print(f"GDS 不可用: {e}")
    duplicates, stats = [], {}
    GDS_AVAILABLE = False

投影创建成功，耗时: 0.68秒
函数 create_entity_projection 执行耗时: 0.68秒
开始检测相似实体...
KNN完成，写入 2 个关系, 用时: 0.37秒
函数 detect_similar_entities 执行耗时: 0.37秒
开始检测社区...
社区检测完成，找到 0 个社区, 用时: 0.10秒
函数 detect_communities 执行耗时: 0.10秒
潜在重复实体查找完成，找到 0 组候选实体, 用时: 0.39秒
函数 find_potential_duplicates 执行耗时: 0.39秒

性能统计摘要:
  总耗时: 1.54秒
  投影时间: 0.68秒 (44.5%)
  KNN时间: 0.37秒 (23.9%)
  WCC时间: 0.10秒 (6.3%)
  查询时间: 0.39秒 (25.3%)
  status: success
  候选实体组数: 0
  关系数量: 2
  社区数量: 0
投影图清理完成
函数 process_entities 执行耗时: 1.72秒
检测完成: 0 组候选重复实体


---
## Step 7：实体合并

In [22]:
if GDS_AVAILABLE and duplicates:
    merger = EntityMerger()
    merged_count, merge_stats = merger.process_duplicates(duplicates)
    print(f"合并完成: {merged_count} 个实体")
else:
    print("跳过合并（无候选组或 GDS 不可用）")

跳过合并（无候选组或 GDS 不可用）


---
## Step 8：消歧 + 对齐

In [23]:
try:
    quality = EntityQualityProcessor()
    quality_result = quality.process()
    QUALITY_OK = True
    print(f"消歧: {quality_result['disambiguated']} 个实体")
    print(f"对齐: {quality_result['aligned']} 个实体")
except Exception as e:
    print(f"质量提升失败: {e}")
    QUALITY_OK = False

开始实体质量提升流程

阶段1: 实体消歧

开始分批处理WCC分组，每批 500 个
策略：每轮查询前500个未处理分组，处理后自动从结果集移除
第 1 轮：查询返回 1 个待处理分组
第 1 轮：已处理完毕，更新 1 个实体
累计：已处理 1 个分组，更新 1 个实体
本轮返回 1 < 500，剩余数据不足一批
第 2 轮：查询返回0个分组，所有数据已处理完毕

消歧完成统计：
  总轮数: 2
  处理的WCC分组数: 1
  更新的实体数: 1

执行最终验证...
验证通过：所有符合条件的WCC分组均已处理


消歧完成: 处理了 1 个实体

              消歧统计               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ 指标                     ┃   数值 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ 设置canonical_id的实体数 │      1 │
│ 耗时                     │ 0.26秒 │
└──────────────────────────┴────────┘

...

发现 0 个需要对齐的canonical组


对齐完成: 合并了 0 个实体

        对齐统计         
┏━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ 指标         ┃   数值 ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━┩
│ 处理的分组数 │      0 │
│ 检测到的冲突 │      0 │
│ 合并的实体数 │      0 │
│ 耗时         │ 0.08秒 │
└──────────────┴────────┘

实体质量提升完成，总耗时: 0.37秒

消歧: 1 个实体
对齐: 0 个实体


---
## Step 9：图谱验证

In [24]:
print("=" * 50)
print("图谱最终统计")
print("=" * 50)

r = graph.query("MATCH (n) RETURN labels(n) AS label, count(n) AS cnt ORDER BY cnt DESC")
for row in r:
    for lbl in row["label"]:
        print(f"  {lbl}: {row['cnt']} 个")

print()
r = graph.query("MATCH ()-[r]->() RETURN type(r) AS t, count(r) AS cnt ORDER BY cnt DESC")
for row in r:
    print(f"  {row['t']}: {row['cnt']} 条")

图谱最终统计
  __Entity__: 6 个
  部件: 6 个
  __Entity__: 3 个
  维修措施: 3 个
  __Chunk__: 2 个
  __Entity__: 2 个
  故障原因: 2 个
  __Entity__: 2 个
  参数: 2 个
  __Document__: 1 个
  __Entity__: 1 个
  设备: 1 个
  __Entity__: 1 个
  故障: 1 个
  __Entity__: 1 个
  故障现象: 1 个

  MENTIONS: 16 条
  包含: 7 条
  导致: 3 条
  维修解决: 3 条
  SIMILAR: 2 条
  PART_OF: 2 条
  关联于: 1 条
  FIRST_CHUNK: 1 条
  NEXT_CHUNK: 1 条
  发生: 1 条
  表现为: 1 条


In [25]:
print("=" * 50)
print("处理汇总")
print("=" * 50)
checks = [
    ("Step 1 文件读取+分块", True),
    ("Step 2 图结构构建", True),
    ("Step 3 实体提取", True),
    ("Step 4 写入数据库", True),
    ("Step 5 Entity Embedding", cnt > 0),
    ("Step 6 相似实体检测", GDS_AVAILABLE),
    ("Step 7 实体合并", not GDS_AVAILABLE or not duplicates or merged_count >= 0),
    ("Step 8 消歧+对齐", not QUALITY_OK or quality_result["disambiguated"] >= 0),
]
for name, ok in checks:
    print(f"  [{'✓' if ok else '✗'}] {name}")

处理汇总
  [✓] Step 1 文件读取+分块
  [✓] Step 2 图结构构建
  [✓] Step 3 实体提取
  [✓] Step 4 写入数据库
  [✓] Step 5 Entity Embedding
  [✓] Step 6 相似实体检测
  [✓] Step 7 实体合并
  [✓] Step 8 消歧+对齐
